$$
\begin{split}
&\delta_t=r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t)\\
&A_t^{\text{GAE}(\gamma,\lambda)}=\sum_{k=t}^{T-1}(\gamma\lambda)^{k-t}\delta_k
\end{split}
$$


```python
with torch.no_grad():
    lastgaelam = 0
    advantages_reversed = []
    gen_len = token_level_rewards.shape[-1]

    for t in reversed(range(gen_len)):
        nextvalues = values[:, t + 1] if t < gen_len - 1 else 0.0
        delta = token_level_rewards[:, t] + gamma * nextvalues - values[:, t]
        lastgaelam = delta + gamma * lam * lastgaelam
        advantages_reversed.append(lastgaelam)
    advantages = torch.stack(advantages_reversed[::-1], dim=1)

    returns = advantages + values
```

$$
A_t=\delta_t+\gamma\lambda A_{t+1}
$$

- $A_t=\delta_t+\gamma\lambda A_{t+1}$
    - $A_{t+1}=\delta_{t+1}+\gamma\lambda A_{t+2}$
    - $A_t=\delta_t+\gamma\lambda(\delta_{t+1}+\gamma\lambda A_{t+2})=\delta_t+\gamma\lambda\delta_{t+1}+(\gamma\lambda)^2A_{t+2}$
    - 展开 $A_{t+2}$，得到 $A_t=\delta_t+\gamma\lambda\delta_{t+1}+(\gamma\lambda)^2\delta_{t+2}+(\gamma\lambda)^3\delta_{t+3}$
    - 最终：$A_t=\delta_t+(\gamma\lambda)\delta_{t+1}+(\gamma\lambda)^2\delta_{t+2}+\cdots+(\gamma\lambda)^{T-t-1}\delta_{T-1}$
- $\lambda = 0$, GAE-target 退化为 TD-target（Actor-critic）方法